## Datos de Aprobación de Crédito Australiano (Statlog Australian Credit Approval)

El conjunto de datos Statlog Australian contiene solicitudes de tarjetas de crédito. Todos los nombres de atributos y valores han sido anonimizados para proteger la confidencialidad. A diferencia de la versión CRX, los atributos categóricos ya están codificados como enteros. El objetivo es predecir si una solicitud es aprobada o rechazada.

**Dataset:** [Statlog Australian Credit Approval - UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/143/statlog+australian+credit+approval)

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2. Carga y Selección de Datos

Se carga el archivo `.dat` separado por espacios, sin encabezado. Los atributos categóricos ya vienen codificados como enteros. No existen valores `?` en este dataset, pero los atributos continuos **A3, A7, A10 y A13** presentan ceros que representan la ausencia de valor y serán imputados con KNN.

| Columna (índice) | Nombre  | Tipo        | Descripción |
|------------------|---------|-------------|-------------|
| 0                | `A1`    | Categórico  | Binario: 0, 1 (antes: a, b) |
| 1                | `A2`    | Continuo    | Atributo continuo |
| 2                | `A3`    | Continuo    | Atributo continuo |
| 3                | `A4`    | Categórico  | 3 clases: 1, 2, 3 (antes: p, g, gg) |
| 4                | `A5`    | Categórico  | 14 clases: 1–14 (antes: ff, d, i, k, j, aa, m, c, w, e, q, r, cc, x) |
| 5                | `A6`    | Categórico  | 9 clases: 1–9 (antes: ff, dd, j, bb, v, n, o, h, z) |
| 6                | `A7`    | Continuo    | Atributo continuo |
| 7                | `A8`    | Categórico  | Binario: 0, 1 (antes: f, t) |
| 8                | `A9`    | Categórico  | Binario: 0, 1 (antes: f, t) |
| 9                | `A10`   | Continuo    | Atributo continuo (entero) |
| 10               | `A11`   | Categórico  | Binario: 0, 1 (antes: f, t) |
| 11               | `A12`   | Categórico  | 3 clases: 1, 2, 3 (antes: s, g, p) |
| 12               | `A13`   | Continuo    | Atributo continuo (entero) |
| 13               | `A14`   | Continuo    | Atributo continuo (entero) |
| 14               | `A15`   | Clase       | **Variable objetivo:** 1 (aprobado) / 0 (rechazado) |

**Variable objetivo (col. 14):** `A15` (decisión de crédito: 1 = aprobado / 0 = rechazado)

In [2]:
# Nombres de columnas (el archivo no tiene encabezado)
column_names = ['A1','A2','A3','A4','A5','A6','A7','A8','A9','A10','A11','A12','A13','A14','A15']

dt = pd.read_csv(
    '../Database/PP7_australian.dat',
    header=None,
    names=column_names,
    sep=' ',
    skipinitialspace=True
)

# Limpiar espacios en nombres de columnas
dt.columns = dt.columns.str.strip()

# Este dataset no tiene columnas a excluir
print(dt.columns.tolist())
# print(pd.isnull(dt).sum())

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15']


In [3]:
# Columnas continuas con posibles ceros problemáticos (ausencia de valor)
# A3, A7, A10, A13 presentan ceros que deben imputarse
# A2 y A14 no tienen ceros → se dejan tal cual
continuous_cols = ['A2', 'A3', 'A7', 'A10', 'A13', 'A14']
categorical_cols = ['A1', 'A4', 'A5', 'A6', 'A8', 'A9', 'A11', 'A12']

# Columnas numéricas para sklearn
numeric_cols = dt.select_dtypes(include=[np.number]).columns.tolist()
# Excluir la variable objetivo del escalado
numeric_cols_feat = [c for c in numeric_cols if c != 'A15']

In [4]:
# Columnas continuas que tienen ceros como valor ausente
zero_impute_cols = ['A3', 'A7', 'A10', 'A13']

# Reemplazar ceros por NaN en esas columnas para aplicar imputación
for col in zero_impute_cols:
    n_zeros = (dt[col] == 0).sum()
    if n_zeros > 0:
        dt[col] = dt[col].replace(0, np.nan)
        print(f"  [{col}] → {n_zeros} ceros reemplazados por NaN")

print(f"\nTotal NaN tras reemplazo de ceros: {dt.isnull().sum().sum()}")

# Escalar columnas de features numéricas (con NaN → usar mediana temporal)
dt_temp = dt[numeric_cols_feat].copy()
for col in dt_temp.columns:
    dt_temp[col] = dt_temp[col].fillna(dt_temp[col].median())

scaler = StandardScaler()
X_num = scaler.fit_transform(dt_temp)

# KNN sobre columnas numéricas (k=5 vecinos)
knn = NearestNeighbors(n_neighbors=6, metric='euclidean')  # 6 porque incluye la propia fila
knn.fit(X_num)
distancias, indices = knn.kneighbors(X_num)

# Imputar columnas continuas con NaN usando KNN-media
for col in zero_impute_cols:
    mask_na = dt[col].isna()
    n_faltantes = mask_na.sum()
    if n_faltantes == 0:
        continue

    print(f"  [{col}] → {n_faltantes} faltantes, imputando con KNN-media...")
    filas_na = dt.index[mask_na].tolist()

    for fila in filas_na:
        vecinos_idx = indices[fila][1:6]
        valores_vecinos = dt[col].iloc[vecinos_idx].dropna()

        if len(valores_vecinos) > 0:
            dt.at[fila, col] = valores_vecinos.mean()
        else:
            dt.at[fila, col] = dt[col].median()

print(f"\nTotal NaN tras imputación KNN: {dt.isnull().sum().sum()}")

print(f"\n✓ Imputación completada.")
print(f"¿Quedan NaN en todo dt? {dt.isnull().sum().sum()}")
print("\nTipos de datos finales:")
print(dt.dtypes.value_counts())
print("\nPrimeras 5 filas:")
print(dt.head())

  [A3] → 19 ceros reemplazados por NaN
  [A7] → 70 ceros reemplazados por NaN
  [A10] → 395 ceros reemplazados por NaN
  [A13] → 132 ceros reemplazados por NaN

Total NaN tras reemplazo de ceros: 616
  [A3] → 19 faltantes, imputando con KNN-media...
  [A7] → 70 faltantes, imputando con KNN-media...
  [A10] → 395 faltantes, imputando con KNN-media...
  [A13] → 132 faltantes, imputando con KNN-media...

Total NaN tras imputación KNN: 0

✓ Imputación completada.
¿Quedan NaN en todo dt? 0

Tipos de datos finales:
int64      10
float64     5
Name: count, dtype: int64

Primeras 5 filas:
   A1     A2     A3  A4  A5  A6     A7  A8  A9   A10  A11  A12    A13   A14  \
0   1  22.08  11.46   2   4   4  1.585   0   0   4.0    1    2  100.0  1213   
1   0  22.67   7.00   2   8   4  0.165   0   0   4.0    0    2  160.0     1   
2   0  29.58   1.75   1   4   4  1.250   0   0   4.0    1    2  280.0     1   
3   0  21.67  11.50   1   5   3  1.342   1   1  11.0    1    2  108.8     1   
4   1  20.17   8.